Scala to Python Converter

In [ ]:
import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
import requests
import traceback

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

# Connect to client libraries

openai = OpenAI()
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)


In [ ]:

models = ["gpt-5", "qwen2.5-coder", "deepseek-coder-v2", "gpt-oss:20b"]

clients = {"gpt-5": openai, "qwen2.5-coder": ollama, "deepseek-coder-v2": ollama, "gpt-oss:20b": ollama}


In [ ]:
import json

# Add course week4/ to path so system_info.py can be imported
_week4 = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if not os.path.isfile(os.path.join(_week4, "system_info.py")):
    _week4 = os.path.abspath(os.path.join(os.getcwd(), "week4"))
if os.path.isfile(os.path.join(_week4, "system_info.py")):
    sys.path.insert(0, _week4)
else:
    raise FileNotFoundError(
        "Could not find week4/system_info.py — open Jupyter from the repo or week4 folder, or cd there once."
    )

from system_info import retrieve_system_info

system_info = retrieve_system_info()

# Saved Python will be syntax-checked and run with the same interpreter as this notebook
OUTPUT_FILE = "main.py"
_py = sys.executable


def _shell_word(path: str) -> str:
    """Quote paths with spaces for use in shell-style command strings."""
    path = os.path.normpath(path)
    if any(c in path for c in " \t\n\"'"):
        return json.dumps(path)
    return path


compile_command = f"{_shell_word(_py)} -m py_compile {_shell_word(OUTPUT_FILE)}"
run_command = f"{_shell_word(_py)} {_shell_word(OUTPUT_FILE)}"

_preview = json.dumps(system_info, indent=2)
print(_preview[:2000] + ("\n…" if len(_preview) > 2000 else ""))
print("compile_command:", compile_command)
print("run_command:", run_command)

In [ ]:
system_prompt = """
Your task is to convert Scala code into high performance Python code.
Respond only with Python code. Do not provide any explanation other than occasional comments.
The python response needs to produce an identical output in the fastest possible time.
"""


def user_prompt_for(scala: str) -> str:
    return f"""
Port this Scala code to Python with the fastest possible implementation that produces identical output in the least time.

System information (JSON):
{json.dumps(system_info, indent=2)}

Your response will be saved as `{OUTPUT_FILE}` and validated with:
- Syntax / bytecode compile: {compile_command}
- Execute: {run_command}

Respond only with Python source code (no markdown fences or prose).
Do not add any additional comments
Scala code to port:

```scala
{scala}
```
"""

In [ ]:
def messages_for(scala):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(scala)}
    ]

def write_output(python):
    with open(OUTPUT_FILE, 'w') as f:
        f.write(python)


In [ ]:

def model_call(model, scala):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(scala), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```python','').replace('```','')
    write_output(reply)
    return reply

def run_python(code):
    write_output(code)
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

import subprocess
import tempfile
import textwrap
from pathlib import Path
from typing import Tuple


def run_scala(code: str) -> Tuple[str, str]:
    """
    Execute an arbitrary Scala snippet and return (stdout, stderr).

    Parameters
    ----------
    code : str
        Plain Scala source.  You may supply either a *full* object definition
        (e.g. `object Foo { … }`) or just a fragment.  If it isn’t a top‑level
        `object`, the helper automatically wraps it as:

            object TempRunner extends App {
                <your code>
            }

    Returns
    -------
    tuple[str, str]
        (stdout, stderr) – `stderr` is an empty string when the run succeeded.

    Notes
    -----
    Requires the `scala` binary on your `$PATH`.  The helper works with both
    Scala 2.x and 3.x.

    Example
    -------
    >>> out, err = run_scala('''
    ...   val x = 10
    ...   println(s"X^2 = ${x*x}")
    ... ''')
    >>> print(out)
    X^2 = 100
    >>> print(err)   # ← empty on success
    """
    # ------------------------------------------------------------------
    # 1.  Wrap ordinary fragments in a minimal App
    # ------------------------------------------------------------------
    code = code.strip()
    if not any(code.startswith(tok) for tok in ("object", "class", "trait", "def")):
        code = f"""
object TempRunner extends App {{
{textwrap.indent(code, '  ')}
}}
"""

    # ------------------------------------------------------------------
    # 2.  Create a temporary directory + file
    # ------------------------------------------------------------------
    tmp_dir = Path(tempfile.mkdtemp())
    src_file = tmp_dir / "TempRunner.scala"
    src_file.write_text(code, encoding="utf-8")


    # ------------------------------------------------------------------
    # 3.  Run the script file with the scala launcher
    # ------------------------------------------------------------------
    result = subprocess.run(
        ["scala", str(src_file)],
        cwd=tmp_dir,                      # keep the cwd tidy
        capture_output=True,
        text=True
    )

    # ------------------------------------------------------------------
    # 4.  Return stdout / stderr (empty string on success)
    # ------------------------------------------------------------------
    if result.returncode != 0:
        return ("", f"Runtime error:\n{result.stderr}")
    return (result.stdout, "")

    



In [ ]:
scala = """

import scala.annotation.tailrec
import java.time.Instant

object MaxSubarrayBenchmark {

  /* ------------------------------------------------------------------- */
  /* 1️⃣  Linear‑congruential generator (infinite iterator)             */
  /* ------------------------------------------------------------------- */
  def lcg(seed: Long,
          a: Long = 1664525L,
          c: Long = 1013904223L,
          m: Long = 1L << 32): Iterator[Long] = new Iterator[Long] {
    var value = seed
    def hasNext: Boolean = true          // infinite stream
    def next(): Long = {
      value = (a * value + c) % m
      value
    }
  }

  /* ------------------------------------------------------------------- */
  /* 2️⃣  Max‑subarray sum (O(n^2) – same as your Python code)          */
  /* ------------------------------------------------------------------- */
  def maxSubarraySum(n: Int,
                     seed: Long,
                     minVal: Int,
                     maxVal: Int): Long = {

    val randGen = lcg(seed)

    // Build the array of random numbers
    val numbers: Array[Int] =
      Array.fill(n)( (randGen.next() % (maxVal - minVal + 1)).toInt + minVal)

    var maxSum = Long.MinValue

    for (i <- 0 until n) {
      var currentSum: Long = 0
      for (j <- i until n) {
        currentSum += numbers(j)
        if (currentSum > maxSum) maxSum = currentSum
      }
    }
    maxSum
  }

  /* ------------------------------------------------------------------- */
  /* 3️⃣  Run the experiment 20 times, summing the results             */
  /* ------------------------------------------------------------------- */
  def totalMaxSubarraySum(n: Int,
                          initialSeed: Long,
                          minVal: Int,
                          maxVal: Int): Long = {
    var total = 0L
    val lcgGen = lcg(initialSeed)

    for (_ <- 0 until 20) {
      val seed = lcgGen.next()
      total += maxSubarraySum(n, seed, minVal, maxVal)
    }
    total
  }

  /* ------------------------------------------------------------------- */
  /* 4️⃣  Main – timings and printing                                   */
  /* ------------------------------------------------------------------- */
  def main(args: Array[String]): Unit = {
    val n            = 10000
    val initialSeed  = 42L
    val minVal       = -10
    val maxVal       = 10

    val start = System.nanoTime()
    val result = totalMaxSubarraySum(n, initialSeed, minVal, maxVal)
    val end = System.nanoTime()

    val elapsedSeconds = (end - start).toDouble / 1e9

    println(s"Total Maximum Subarray Sum (20 runs): $result")
    println(f"Execution Time: $elapsedSeconds%.6f seconds")
    elapsedSeconds
  }
}
"""



In [305]:
with gr.Blocks() as ui:
    with gr.Row():
        scala_inp  = gr.Textbox(label="Scala code:", lines=50, value=scala)
        python_inp = gr.Textbox(label="Python code:", lines=50)

    with gr.Row():
        run_scala_btn = gr.Button("Run Scala")
        output_scala   = gr.TextArea(label="Scala Output", lines=10)

        run_python_btn = gr.Button("Run Python")
        output_python  = gr.Textbox(label="Python Output", lines=10)

    with gr.Row():
        model = gr.Dropdown(
            choices=models,
            label="Select model",
            value=models[0]
        )
        convert_btn = gr.Button("Convert code")

    # ---- attach events --------------------------------------
    convert_btn.click(
        fn=model_call,
        inputs=[model, scala_inp],
        outputs=[python_inp]
    )

    run_scala_btn.click(
        fn=run_scala,
        inputs=[scala_inp],
        outputs=[output_scala]
    )

    run_python_btn.click(
        fn=run_python,
        inputs=[python_inp],
        outputs=[output_python]
    )

# ----------  launch ----------
ui.launch(inbrowser=True, debug=True)

Keyboard interruption in main thread... closing server.
